# 01 · Choose and import a dataset

**Lecture 1 · Block B1 · Lab · 10 minutes**

**Learning goal:** We can describe a dataset as *objects × attributes* and name the scale type of every attribute.

**What we do:**
1. Look at the four course datasets and pick one per group.
2. Write down the question we want to answer (Task 1).
3. Classify every column as numeric, ordinal, nominal, binary or ignore (Task 2).

Loading is the easy part: the `ddm` helper library does it for us. The tricky part is the classification. Is `region = 3` a number? Is the year of measurement a property of a penguin? There is not always one right answer, but there are wrong ones. Our classification is saved at the end and used by `02_distance.ipynb`.

In [ ]:
DATASET = "penguins"  # or "wholesale", "auto_mpg", "students" (see 01_dataset.ipynb)

In [ ]:
import pandas as pd

import ddm

pd.set_option("display.max_columns", 40)

## Step 1 · The four course datasets

All four have 300–450 objects, attributes of different types, and at least one
*label candidate*: a column with **known groups** (e.g. `species`). We do **not** cluster on it,
we only compare our clusters with it at the end.

In [ ]:
ddm.list_datasets()

## Step 2 · Load the data and look at it

In [ ]:
df = ddm.load_dataset(DATASET)
info = ddm.dataset_info(DATASET)

print(f"{info.title}: {df.shape[0]} objects (rows) x {df.shape[1]} attributes (columns)")
print(f"Question of the dataset: {info.question}")
print(f"Label candidates (known groups, NOT features): {', '.join(info.label_candidates)}")
print("Pitfalls:")
for pitfall in info.pitfalls:
    print("  -", pitfall)
df.head()

In [ ]:
ddm.numeric_summary(df)

`dtype` is how pandas *stores* a column, not its scale type. An `int64` column can be a
nominal code (for example `origin = 1, 2, 3` for USA, Europe, Japan), and `str` can be
ordinal (small < medium < large).

## Task 1 · Our question (easy, 2 minutes)

One sentence: what do we want to find out with clustering? Start from the question of the
dataset above and make it our own.

In [ ]:
# TODO: one sentence
QUESTION = "..."

## Task 2 · Classify every attribute (the tricky part, 6 minutes)

| Type | Meaning | Meaningful operations | Example |
|------|---------|-----------------------|---------|
| `numeric` | interval or ratio scale: differences are meaningful | =, <, − (ratio scale also ÷) | body mass in g |
| `ordinal` | ordered categories, the steps are not equally large | =, < | education level 0–4 |
| `nominal` | categories without order, **also when stored as integer codes** | = | island, region code |
| `binary` | exactly two states | = | yes / no |
| `ignore` | not a feature: IDs, free text, the label candidates, how/when the data was recorded | | species, car name |

<details>
<summary><b>Hint:</b> three questions per column (click to open)</summary>

1. Does the column describe the object itself, or how/when it was recorded, or does it hold the known groups? → `ignore`
2. Is a difference of two values meaningful (5 g more, 3 years older)? → `numeric`
3. Is there an order? Yes → `ordinal`. No → `nominal`, or `binary` if there are only two values.

</details>

The next cell prints a template. Copy it into the cell after it and replace every `"?"`.

In [ ]:
# Given: print a template for TYPES
print("TYPES = {")
for col in df.columns:
    example = df[col].dropna().iloc[0]
    example = example.item() if hasattr(example, "item") else example  # numpy number -> plain number
    print(f'    "{col}": "?",  # {df[col].dtype}, {df[col].nunique()} distinct, e.g. {example!r}')
print("}")

In [ ]:
# TODO: paste the template printed above and replace every "?" by
#       "numeric", "ordinal", "nominal", "binary" or "ignore".
TYPES = None

In [ ]:
# Given: check the classification and save it for 02_distance.ipynb
VALID_TYPES = {"numeric", "ordinal", "nominal", "binary", "ignore"}


def check_types(types: dict, df: pd.DataFrame) -> list[str]:
    """Return a list of problems (empty = fine) and print hints for doubtful choices."""
    problems = []
    missing = [c for c in df.columns if c not in types]
    unknown = [c for c in types if c not in df.columns]
    invalid = {c: t for c, t in types.items() if t not in VALID_TYPES}
    if missing:
        problems.append(f"columns without a type: {missing}")
    if unknown:
        problems.append(f"columns that do not exist: {unknown}")
    if invalid:
        problems.append(f"invalid types: {invalid}")
    if len([c for c, t in types.items() if t != "ignore"]) < 2:
        problems.append("use at least two attributes")

    for c in info.label_candidates:
        if types.get(c, "ignore") != "ignore":
            print(f"HINT  '{c}' is a label candidate. If it is a feature, comparing the clusters with it later is circular.")
    for c, t in types.items():
        if c not in df.columns:
            continue
        if t == "numeric" and pd.api.types.is_integer_dtype(df[c]) and df[c].nunique() <= 5:
            print(f"HINT  '{c}' is 'numeric' but has only {df[c].nunique()} distinct integers: a code, an order or a count?")
        if t == "binary" and df[c].nunique() != 2:
            print(f"HINT  '{c}' is 'binary' but has {df[c].nunique()} distinct values.")
    return problems


if TYPES is None:
    raise NotImplementedError("Task 2: fill in TYPES in the cell above.")

problems = check_types(TYPES, df)
if problems:
    for problem in problems:
        print("FAIL ", problem)
else:
    print("PASS  every column has a valid type:", pd.Series(TYPES).value_counts().to_dict())
    path = ddm.save_json(DATASET, "types", TYPES)
    print(f"Saved to {path}")

## Reflection (2 minutes)

1. Which column was the hardest to classify, and why?
2. Look at the value ranges below. If we computed a plain Euclidean distance on the raw
   values, which attribute would decide almost everything?

In [ ]:
df.describe()